# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata

print(f"\nDataset Name: {metadata_obj.name}")
print(f"Description: {metadata_obj.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs. We'll enumerate all record sets and show their `@id` and contained fields (and their `@id`) within the dataset package.

In [ ]:
from pprint import pprint

record_sets = dataset.record_sets
print(f"\nTotal record sets found: {len(record_sets)}\n")
overview = []
for rs in record_sets:
    print(f"Record Set: {rs.id}")
    print(f"  name: {getattr(rs, 'name', 'N/A')}")
    # fields may be an attribute or empty
    if hasattr(rs, 'fields') and rs.fields:
        for fld in rs.fields:
            print(f"    Field: {fld.id}  (name: {getattr(fld, 'name', '')})")
    elif hasattr(rs, 'columns') and rs.columns:
        for col in rs.columns:
            print(f"    Column: {col.id}  (name: {getattr(col, 'name', '')})")
    else:
        print("    No fields or columns defined.")
    print()
    overview.append(rs.id)

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. We use the record set `@id` fields from the overview above.

In [ ]:
# Build a dictionary of DataFrame for each record set using its @id
dfs = {}
for rs in record_sets:
    rsid = rs.id
    print(f"Loading records for record set: {rsid}")
    try:
        records = list(dataset.records(record_set=rsid))
        if records:
            dfs[rsid] = pd.DataFrame(records)
            print(f"Loaded {len(dfs[rsid])} records for {rsid}.")
        else:
            print(f"No records found for {rsid}.")
    except Exception as e:
        print(f"Error loading {rsid}: {e}")


# Display available dataframes and their columns
for key, df in dfs.items():
    print(f"\nDataFrame for record set @id: {key}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps to one record set. Below, we pick the first available record set with at least one numeric field. All fields (columns) are referenced by their `@id`.

* Selecting a numeric field for outlier filtering and normalization
* Filtering records with values above a threshold
* Normalizing the field
* Optionally grouping by a categorical field

In [ ]:
# Choose a record set with at least one numeric field
import numpy as np

# Helper to find a numeric field
chosen_rsid = None
numeric_field_id = None

for rs in record_sets:
    rsid = rs.id
    if rsid not in dfs or dfs[rsid].empty:
        continue
    df = dfs[rsid]

    # Try columns, then fields, referencing only @id
    possible_candidates = []
    # Croissant allows either 'columns' or 'fields'
    if hasattr(rs, 'columns') and rs.columns:
        for col in rs.columns:
            field_id = col.id
            # Try to guess numeric by dtype
            if field_id in df.columns and pd.api.types.is_numeric_dtype(df[field_id]):
                possible_candidates.append(field_id)
    elif hasattr(rs, 'fields') and rs.fields:
        for fld in rs.fields:
            field_id = fld.id
            if field_id in df.columns and pd.api.types.is_numeric_dtype(df[field_id]):
                possible_candidates.append(field_id)
    if possible_candidates:
        chosen_rsid = rsid
        numeric_field_id = possible_candidates[0]
        break

if not chosen_rsid or not numeric_field_id:
    print("No suitable numeric field found for EDA in any record set.")
else:
    print(f"Using record set @id: {chosen_rsid}\nUsing numeric field @id: {numeric_field_id}")
    df = dfs[chosen_rsid]
    # Drop NA for the numeric field to simplify processing
    filtered_df = df[df[numeric_field_id].notna()]

    # Set a threshold as the 75th percentile (or fallback to 10 if range is small)
    threshold = np.percentile(filtered_df[numeric_field_id], 75) if len(filtered_df) > 0 else 10

    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization (z-score)
    filtered_df[numeric_field_id + "_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

    # Try to group by a categorical field from the same record set
    # We'll look for a string/categorical column not identical to the numeric
    group_field = None
    for col in filtered_df.columns:
        if col == numeric_field_id or col.endswith("_normalized"):
            continue
        if pd.api.types.is_object_dtype(filtered_df[col]):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped means of {numeric_field_id} by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we plot the distribution of the numeric field and, if a grouping field is found, mean values by group.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure at least one EDA field was loaded
if chosen_rsid and numeric_field_id and chosen_rsid in dfs and not dfs[chosen_rsid].empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(dfs[chosen_rsid][numeric_field_id], kde=True, bins=30)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # Grouped bar plot if we found a group_field
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df)
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we've loaded the FAIR^2 dataset package, enumerated its record sets, explored the fields using Croissant `@id`s, and performed preliminary EDA including filtering and normalization of numeric fields. Visual summary plots help to understand the data distributions and grouping patterns. For further analysis, consult field documentation in the Croissant schema and extend the EDA to additional record sets and fields.